In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Dataset
import h5py
import wandb
import numpy as np

In [6]:
 # Load data
BATCH_SIZE = 512
NUM_WORKERS = 0
filename = r"train_test_split.h5"

def quarter(pattern):
    return pattern[:32, :32]  # downsample to 32x32

# Custom Dataset class
class NeffDataset(Dataset):
    def __init__(self, patterns, params, targets):
        self.patterns = torch.FloatTensor(patterns)
        self.params = torch.FloatTensor(params)
        self.targets = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.patterns)
    
    def __getitem__(self, idx):
        return self.patterns[idx], self.params[idx], self.targets[idx]
    
def load_data(filename):
    """Load and process data from HDF5 file"""
    with h5py.File(filename, "r") as f:
        neff_train = np.array(f['neff_train'])
        weight_train = np.array(f['weight_train'])
        params_train = np.array(f['params_train'])
        pattern_train = np.array(f['pattern_train'])  # shape [N, 64, 64]
        
        neff_test = np.array(f['neff_test'])
        weight_test = np.array(f['weight_test'])
        params_test = np.array(f['params_test'])
        pattern_test = np.array(f['pattern_test'])  # shape [M, 64, 64]
    
    # Apply quarter() to every pattern (vectorized alternative)
    pattern_train_quartered = pattern_train[:, :32, :32]  # shape [N, 32, 32]
    pattern_test_quartered  = pattern_test[:, :32, :32]   # shape [M, 32, 32]

    # Reshape for downstream use if needed
    xs  = pattern_train_quartered[:, np.newaxis, :, :]
    xs1 = pattern_test_quartered[:,  np.newaxis, :, :]

    x1s  = params_train
    x1s1 = params_test
    ys   = neff_train[:, 0:1]
    ys1  = neff_test[:, 0:1]
    
    return xs, x1s, ys, xs1, x1s1, ys1

def create_dataloaders(xs, x1s, ys, xs1, x1s1, ys1, batch_size=256, num_workers=0):
    """Create training and test dataloaders with Windows-compatible settings"""
    train_dataset = NeffDataset(xs, x1s, ys)
    test_dataset = NeffDataset(xs1, x1s1, ys1)
    
    # Use single process for Windows compatibility
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers,  # Use 0 for Windows compatibility
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers,  # Use 0 for Windows compatibility
        pin_memory=True
    )
    
    return train_loader, test_loader


try:
    xs, x1s, ys, xs1, x1s1, ys1 = load_data(filename)
    print(f"Training data shape: {xs.shape}")
    print(f"Training params shape: {x1s.shape}")
    print(f"Training targets shape: {ys.shape}")
    print(f"Test data shape: {xs1.shape}")
    print(f"Test params shape: {x1s1.shape}")
    print(f"Test targets shape: {ys1.shape}")
except FileNotFoundError:
    print(f"Data file not found: {filename}")
    print("Please update the filename path")
    exit()

# Create dataloaders
train_loader, test_loader = create_dataloaders(xs, x1s, ys, xs1, x1s1, ys1, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

Training data shape: (1113998, 1, 32, 32)
Training params shape: (1113998, 4)
Training targets shape: (1113998, 1)
Test data shape: (477427, 1, 32, 32)
Test params shape: (477427, 4)
Test targets shape: (477427, 1)
Number of training batches: 2176
Number of test batches: 933


In [ ]:
import time
import torch
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def profile_dataloader(loader, device=device, warmup_batches=5, test_batches=50):
    """
    Profiles dataloader performance over several batches.
    Measures:
      - Data load time (CPU)
      - Host→Device copy time
      - Total batch time
      - Effective samples/sec
    """

    data_times, transfer_times, total_times = [], [], []
    n_samples = 0

    # Warmup (fills caches)
    print("Warming up...")
    for i, (x, p, y) in enumerate(loader):
        if i >= warmup_batches:
            break

    torch.cuda.synchronize() if device.type == "cuda" else None
    print("Running timed iterations...")

    start_all = time.time()
    for i, (x, p, y) in enumerate(tqdm(loader, total=test_batches, desc="Profiling")):
        if i >= test_batches:
            break

        # Time each stage
        t0 = time.time()
        # (Data already loaded by DataLoader)
        t1 = time.time()

        # Host → Device copy
        x = x.to(device, non_blocking=True)
        p = p.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        torch.cuda.synchronize() if device.type == "cuda" else None
        t2 = time.time()

        total_times.append(t2 - t0)
        data_times.append(t1 - t0)
        transfer_times.append(t2 - t1)
        n_samples += x.size(0)

    total_time = time.time() - start_all
    torch.cuda.synchronize() if device.type == "cuda" else None

    print("\n===== DataLoader Profiling Report =====")
    print(f"Device: {device}")
    print(f"Total samples processed: {n_samples}")
    print(f"Total elapsed time: {total_time:.2f} s")

    def stats(arr): return np.median(arr), np.mean(arr), np.std(arr)

    dt_med, dt_mean, dt_std = stats(data_times)
    tt_med, tt_mean, tt_std = stats(transfer_times)
    tot_med, tot_mean, tot_std = stats(total_times)

    print(f"\nPer-batch timings (median ± std, seconds):")
    print(f"  Data load (CPU):     {dt_med:.4f} ± {dt_std:.4f}")
    print(f"  Host→Device copy:    {tt_med:.4f} ± {tt_std:.4f}")
    print(f"  Total per batch:     {tot_med:.4f} ± {tot_std:.4f}")
    print(f"\nEffective throughput:")
    print(f"  {n_samples / total_time:.2f} samples/sec overall")

    return {
        "data_load_times": data_times,
        "transfer_times": transfer_times,
        "total_times": total_times,
        "samples_per_sec": n_samples / total_time,
    }

# ---- Run the diagnostic ----
stats = profile_dataloader(train_loader)
